<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase53.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 53 — authorized real-source → private FHIR R4 Bulk NDJSON with explicit missingness

Phase 52 froze an authorized private longitudinal neuroimaging source reference:

- 258 source persons;
- 4,128 normalized longitudinal measurement rows;
- 258 persons with complete bilateral hippocampal baseline→M12 evidence;
- 258 persons with complete bilateral entorhinal baseline→M12 evidence.

However, the selected source does **not** establish explicit measurement units, acquisition/DICOM identifiers, model version, provenance identifiers, or clinical MCI status.

Phase 53 constructs a deterministic **private FHIR R4 source representation** without inventing those missing semantics.

### FHIR representation policy

- `Patient`: deterministic private FHIR ID only; no fabricated gender or birth date.
- `Observation.status`: `unknown` because source finality is not established.
- `Observation.code`: local source-column token coding, not a claimed standard terminology concept.
- `Observation.bodySite`: local anatomy/laterality coding.
- `Observation.valueQuantity.value`: source numeric value.
- Unit fields are omitted when the source unit is not explicit.
- Baseline/M12 are represented as local source-timepoint extensions; no fabricated clinical dates.
- No `ImagingStudy`, `Device`, or `Provenance` resource is created when the required source identifiers are absent.
- No diagnosis/Condition resource is created because clinical MCI status is absent.

This phase creates a private FHIR source artifact. It does **not** execute OMOP/MI-CDM transformation or the proposed fidelity method.


In [1]:
# CODE CELL 1/8 — Drive, constants, helpers
import os, re, csv, json, gzip, hashlib, tempfile, shutil, zipfile, time
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='53_authorized_real_source_to_private_fhir_r4_bulk_ndjson'
EXPECTED_P52_SHAREABLE='9750a0614b57d94ebcd68161a76ddad23743becd9aa5c99b910ffb08a74023f1'
SOURCE_TOKEN='069caf1537e082c6'
EXPECTED_SOURCE_PERSONS=258
EXPECTED_NORMALIZED_ROWS=4128

FHIR_ID_RE=re.compile(r'^[A-Za-z0-9\-.]{1,64}$')
LOCAL_CODE_SYSTEM='urn:ichi2027:source-column-token'
LOCAL_BODY_SITE_SYSTEM='urn:ichi2027:anatomy-laterality'
EXT_TIMEPOINT='urn:ichi2027:extension:source-timepoint'
EXT_UNIT_EVIDENCE='urn:ichi2027:extension:unit-evidence'
EXT_SOURCE_FILE='urn:ichi2027:extension:source-file-token'
EXT_SOURCE_COLUMN='urn:ichi2027:extension:source-column-token'

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def canonical_sha(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode('utf-8')).hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

if os.environ.get('PHASE53_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE53_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE53_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase53_'))
STAGE=WORK/'public_outputs'; STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE53_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase53')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

PRIVATE_ROOT=MYDRIVE/'ICHI2027'/'_PRIVATE_REAL_MRI'
PRIVATE_PHASE52=PRIVATE_ROOT/'Phase52'
PRIVATE_RUN=PRIVATE_ROOT/'Phase53'/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
PRIVATE_RUN.mkdir(parents=True,exist_ok=False)

print('Phase 53 public output:',OUT)
print('Private FHIR workspace prepared.')


Mounted at /content/drive
Phase 53 public output: /content/drive/MyDrive/ICHI2027/Phase53/run_20261001T053521_159753Z
Private FHIR workspace prepared.


In [2]:
# CODE CELL 2/8 — Verify exact Phase52 public evidence and locate matching private freeze

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

hits=[]
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            if name_matches(fn,'ICHI2027_Phase52_SHAREABLE','.json'):
                p=Path(here)/fn
                try:
                    if sha256(p)==EXPECTED_P52_SHAREABLE:hits.append(p)
                except OSError:pass

if not hits:
    raise FileNotFoundError('Exact Phase52 SHAREABLE.json was not found in Drive.')

P52=sorted(hits,key=str)[0]
p52=json.loads(P52.read_text('utf-8'))

assert p52['phase']=='52_authorized_real_source_normalization_and_private_evidence_freeze'
assert p52['authorization_attestation_valid'] is True
assert p52['private_source_manifest_created'] is True
assert p52['private_normalized_source_created'] is True
assert p52['selected_source_token']==SOURCE_TOKEN
assert p52['source_unique_persons']==EXPECTED_SOURCE_PERSONS
assert p52['normalized_longitudinal_measurement_rows']==EXPECTED_NORMALIZED_ROWS
assert p52['persons_complete_bilateral_hippocampus_baseline_m12']==258
assert p52['ready_for_private_fhir_source_representation'] is True
assert p52['ready_for_official_armA'] is False

# Select a valid private Phase52 run by reading its private manifest.
private_candidates=[]
if PRIVATE_PHASE52.is_dir():
    for run in sorted(PRIVATE_PHASE52.glob('run_*'),reverse=True):
        mf=run/'ICHI2027_Phase52_PRIVATE_SOURCE_MANIFEST.json'
        if not mf.is_file():continue
        try:
            m=json.loads(mf.read_text('utf-8'))
            if (
                m.get('source_file_token')==SOURCE_TOKEN
                and int(m.get('source_rows',-1))==258
                and m.get('authorization_attestation',{}).get('authorized_access') is True
                and m.get('authorization_attestation',{}).get('authorized_for_local_research_analysis') is True
            ):
                norm_candidates=[
                    run/'ICHI2027_Phase52_NORMALIZED_SOURCE_PRIVATE.parquet',
                    run/'ICHI2027_Phase52_NORMALIZED_SOURCE_PRIVATE.csv.gz'
                ]
                norm=next((x for x in norm_candidates if x.is_file()),None)
                if norm is not None:
                    private_candidates.append((run,mf,norm,m))
        except Exception:
            pass

if not private_candidates:
    raise FileNotFoundError('No matching authorized private Phase52 normalized-source freeze was found.')

P52_PRIVATE_RUN,P52_PRIVATE_MANIFEST,NORMALIZED,private52=private_candidates[0]

print('PASS: Phase52 public/private evidence chain verified.')
print('Private normalized source located by source token and authorization state.')


PASS: Phase52 public/private evidence chain verified.
Private normalized source located by source token and authorization state.


In [3]:
# CODE CELL 3/8 — Read and validate private normalized source

import pandas as pd
import numpy as np

if NORMALIZED.suffix=='.parquet':
    src=pd.read_parquet(NORMALIZED)
else:
    src=pd.read_csv(NORMALIZED)

required=[
 'person_key_private','source_file_token','source_column_token',
 'anatomy','laterality','timepoint','value_numeric','unit','unit_evidence',
 'acquisition_identifier','model_version','provenance_identifier','diagnostic_status'
]
missing=[c for c in required if c not in src.columns]
if missing:raise RuntimeError('Private normalized source missing required columns: '+str(missing))

assert len(src)==EXPECTED_NORMALIZED_ROWS
assert src['person_key_private'].nunique()==EXPECTED_SOURCE_PERSONS
assert set(src['source_file_token'].astype(str))=={SOURCE_TOKEN}
assert set(src['timepoint'].astype(str))<= {'BASELINE','M12'}
assert src['value_numeric'].notna().all()

def nonempty_count(series):
    return int(series.fillna('').astype(str).str.strip().ne('').sum())

explicit_unit_rows=int(src['unit_evidence'].fillna('').eq('EXPLICIT_IN_COLUMN_NAME').sum())
unit_value_rows=nonempty_count(src['unit'])
acquisition_rows=nonempty_count(src['acquisition_identifier'])
model_rows=nonempty_count(src['model_version'])
provenance_rows=nonempty_count(src['provenance_identifier'])
diagnosis_rows=nonempty_count(src['diagnostic_status'])

# Cross-check Phase52 public missingness claims.
assert explicit_unit_rows==int(p52['explicit_unit_rows'])==0
assert acquisition_rows==0 and p52['acquisition_identifier_available'] is False
assert model_rows==0 and p52['model_version_available'] is False
assert provenance_rows==0 and p52['provenance_identifier_available'] is False
assert diagnosis_rows==0 and p52['clinical_mci_status_available'] is False

print('PASS: private normalized source cardinality and missingness claims reproduced.')
print('Rows:',len(src),'Persons:',src['person_key_private'].nunique())


PASS: private normalized source cardinality and missingness claims reproduced.
Rows: 4128 Persons: 258


In [4]:
# CODE CELL 4/8 — Deterministically construct private FHIR R4 resources

def fhir_id(prefix,*parts):
    blob='|'.join([prefix]+[str(x) for x in parts])
    return hashlib.sha256(blob.encode('utf-8')).hexdigest()[:32]

def patient_resource(person_key):
    rid=fhir_id('Patient',person_key)
    return {
      'resourceType':'Patient',
      'id':rid
    }

def observation_resource(row):
    pid=fhir_id('Patient',row.person_key_private)
    oid=fhir_id(
      'Observation',row.person_key_private,row.source_column_token,
      row.anatomy,row.laterality,row.timepoint
    )
    body_code=row.anatomy if not str(row.laterality).strip() else f'{row.anatomy}-{row.laterality}'
    obs={
      'resourceType':'Observation',
      'id':oid,
      'status':'unknown',
      'code':{
        'coding':[{
          'system':LOCAL_CODE_SYSTEM,
          'code':str(row.source_column_token)
        }]
      },
      'subject':{'reference':f'Patient/{pid}'},
      'bodySite':{
        'coding':[{
          'system':LOCAL_BODY_SITE_SYSTEM,
          'code':body_code
        }]
      },
      'valueQuantity':{
        'value':float(row.value_numeric)
      },
      'extension':[
        {'url':EXT_TIMEPOINT,'valueCode':str(row.timepoint)},
        {'url':EXT_UNIT_EVIDENCE,'valueCode':str(row.unit_evidence)},
        {'url':EXT_SOURCE_FILE,'valueString':SOURCE_TOKEN},
        {'url':EXT_SOURCE_COLUMN,'valueString':str(row.source_column_token)}
      ]
    }

    unit=str(row.unit).strip() if pd.notna(row.unit) else ''
    if unit:
        obs['valueQuantity']['unit']=unit
        # Do not invent UCUM system/code unless the source explicitly supplied it.
    return obs

people=sorted(src['person_key_private'].astype(str).unique())
patients=[patient_resource(p) for p in people]
observations=[observation_resource(r) for r in src.itertuples(index=False)]

assert len(patients)==258
assert len(observations)==4128
assert len({p['id'] for p in patients})==258
assert len({o['id'] for o in observations})==4128

# Determinism test: rebuild a second time and compare canonical hashes.
patients2=[patient_resource(p) for p in people]
observations2=[observation_resource(r) for r in src.itertuples(index=False)]
assert canonical_sha(patients)==canonical_sha(patients2)
assert canonical_sha(observations)==canonical_sha(observations2)

print('PASS: deterministic private FHIR resources constructed.')
print('Patients:',len(patients),'Observations:',len(observations))


PASS: deterministic private FHIR resources constructed.
Patients: 258 Observations: 4128


In [5]:
# CODE CELL 5/8 — Structural FHIR R4 preflight and source-preservation checks

patient_ids={p['id'] for p in patients}
validation=[]

def add(check,passed,evidence):
    validation.append({'check':check,'passed':bool(passed),'evidence':evidence})

add('patient_count_258',len(patients)==258,str(len(patients)))
add('observation_count_4128',len(observations)==4128,str(len(observations)))
add('all_patient_ids_fhir_id_lexical',all(FHIR_ID_RE.fullmatch(p['id']) for p in patients),'FHIR id regex')
add('all_observation_ids_fhir_id_lexical',all(FHIR_ID_RE.fullmatch(o['id']) for o in observations),'FHIR id regex')
add('all_observation_status_unknown',all(o['status']=='unknown' for o in observations),'No source finality invented')
add('all_subject_references_resolve',
    all(o['subject']['reference'].split('/',1)[1] in patient_ids for o in observations),
    'Observation.subject -> Patient')
add('all_observations_have_numeric_value',
    all(isinstance(o['valueQuantity']['value'],(int,float)) for o in observations),
    'valueQuantity.value')
add('no_units_invented',
    all('unit' not in o['valueQuantity'] and 'system' not in o['valueQuantity'] and 'code' not in o['valueQuantity']
        for o in observations),
    'Phase52 explicit unit rows = 0')
add('no_effective_datetime_invented',
    all(not any(k.startswith('effective') for k in o) for o in observations),
    'Source has baseline/M12 labels but no clinical dates')
add('no_imagingstudy_resources_created',True,'No acquisition/study/series identifiers available')
add('no_device_resources_created',True,'No model/version source evidence')
add('no_provenance_resources_created',True,'No provenance source identifier')
add('no_condition_resources_created',True,'No clinical MCI source status')
add('all_timepoint_extensions_present',
    all(any(e.get('url')==EXT_TIMEPOINT and e.get('valueCode') in {'BASELINE','M12'} for e in o['extension'])
        for o in observations),
    'Local source-timepoint extension')
add('all_source_column_tokens_preserved',
    all(any(e.get('url')==EXT_SOURCE_COLUMN and e.get('valueString') for e in o['extension'])
        for o in observations),
    'Opaque local source-column token')

if not all(r['passed'] for r in validation):
    raise RuntimeError('FHIR structural preflight failed: '+str([r for r in validation if not r['passed']]))

print('PASS: all FHIR structural/preservation checks passed.')


PASS: all FHIR structural/preservation checks passed.


In [6]:
# CODE CELL 6/8 — Freeze private Bulk NDJSON and private FHIR manifest

PATIENT_NDJSON=PRIVATE_RUN/'Patient.ndjson'
OBS_NDJSON=PRIVATE_RUN/'Observation.ndjson'

with PATIENT_NDJSON.open('w',encoding='utf-8') as f:
    for r in patients:
        f.write(json.dumps(r,sort_keys=True,separators=(',',':'),ensure_ascii=False)+'\n')

with OBS_NDJSON.open('w',encoding='utf-8') as f:
    for r in observations:
        f.write(json.dumps(r,sort_keys=True,separators=(',',':'),ensure_ascii=False)+'\n')

# Readback parse and count.
def read_ndjson(path):
    with Path(path).open('r',encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

rp=read_ndjson(PATIENT_NDJSON)
ro=read_ndjson(OBS_NDJSON)
assert len(rp)==258 and len(ro)==4128
assert canonical_sha(rp)==canonical_sha(patients)
assert canonical_sha(ro)==canonical_sha(observations)

PRIVATE_FHIR_MANIFEST={
 'project':PROJECT,
 'phase':PHASE,
 'created_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'source_file_token':SOURCE_TOKEN,
 'phase52_private_manifest_path':str(P52_PRIVATE_MANIFEST),
 'phase52_normalized_source_path':str(NORMALIZED),
 'phase52_normalized_source_sha256':sha256(NORMALIZED),
 'patient_ndjson_path':str(PATIENT_NDJSON),
 'patient_ndjson_sha256':sha256(PATIENT_NDJSON),
 'observation_ndjson_path':str(OBS_NDJSON),
 'observation_ndjson_sha256':sha256(OBS_NDJSON),
 'patient_count':258,
 'observation_count':4128,
 'fhir_version':'R4 representation policy',
 'coding_policy':'LOCAL_OPAQUE_SOURCE_TOKENS_NO_STANDARD_CONCEPT_CLAIM',
 'observation_status_policy':'unknown because source finality is not established',
 'unit_policy':'omit unit/system/code when source unit is not explicit',
 'time_policy':'local baseline/M12 extension; no fabricated dates',
 'omitted_resource_types':{
   'ImagingStudy':'source acquisition/study/series identifier absent',
   'Device':'source model/version absent',
   'Provenance':'source provenance identifier absent',
   'Condition':'clinical MCI/source diagnosis absent'
 },
 'structural_validation_all_passed':all(r['passed'] for r in validation)
}
PRIVATE_MANIFEST=PRIVATE_RUN/'ICHI2027_Phase53_PRIVATE_FHIR_MANIFEST.json'
PRIVATE_MANIFEST.write_text(json.dumps(PRIVATE_FHIR_MANIFEST,indent=2)+'\n',encoding='utf-8')

print('PASS: private FHIR Bulk NDJSON frozen and read back.')


PASS: private FHIR Bulk NDJSON frozen and read back.


In [7]:
# CODE CELL 7/8 — Public aggregate preservation/missingness evidence and next gate

coverage=[]
for (a,side,tp),g in src.groupby(['anatomy','laterality','timepoint'],dropna=False):
    coverage.append({
      'anatomy':str(a),
      'laterality':'' if pd.isna(side) else str(side),
      'timepoint':str(tp),
      'source_measurement_rows':len(g),
      'source_unique_persons':g['person_key_private'].nunique(),
      'fhir_observation_rows':len(g),
      'numeric_value_row_count_preserved':True
    })

missingness=[
 {'dimension':'gender','source_status':'ABSENT','fhir_action':'Patient.gender omitted','invented':False},
 {'dimension':'birth_date','source_status':'ABSENT','fhir_action':'Patient.birthDate omitted','invented':False},
 {'dimension':'measurement_unit','source_status':'NOT_EXPLICIT','fhir_action':'Quantity.unit/system/code omitted','invented':False},
 {'dimension':'clinical_date','source_status':'ABSENT','fhir_action':'Observation.effective[x] omitted; local timepoint extension retained','invented':False},
 {'dimension':'acquisition_or_dicom_id','source_status':'ABSENT','fhir_action':'No ImagingStudy resource created','invented':False},
 {'dimension':'model_version','source_status':'ABSENT','fhir_action':'No Device/model reference created','invented':False},
 {'dimension':'provenance_identifier','source_status':'ABSENT','fhir_action':'No Provenance resource created','invented':False},
 {'dimension':'clinical_mci_status','source_status':'ABSENT','fhir_action':'No Condition/diagnosis assertion created','invented':False}
]

preservation=[
 {'metric':'source_persons','source_count':258,'fhir_count':258,'exact_count_preserved':True},
 {'metric':'normalized_measurement_rows','source_count':4128,'fhir_count':4128,'exact_count_preserved':True},
 {'metric':'subject_reference_resolution','source_count':4128,'fhir_count':4128,'exact_count_preserved':True},
 {'metric':'numeric_values_represented','source_count':4128,'fhir_count':4128,'exact_count_preserved':True},
 {'metric':'timepoint_labels_represented','source_count':4128,'fhir_count':4128,'exact_count_preserved':True},
 {'metric':'source_column_tokens_represented','source_count':4128,'fhir_count':4128,'exact_count_preserved':True}
]

# FHIR source exists, but official Arm A remains blocked because PERSON/MI-CDM target requirements
# cannot be satisfied honestly from this source alone.
ready_for_fhir_to_target_preflight=True
ready_for_official_armA=False
next_step='PHASE54_COMPANION_SOURCE_SEARCH_FOR_DEMOGRAPHICS_UNITS_ACQUISITION_AND_PROVENANCE'

print('Real private FHIR source representation ready:',ready_for_fhir_to_target_preflight)
print('Official Arm A ready:',ready_for_official_armA)
print('Next step:',next_step)


Real private FHIR source representation ready: True
Official Arm A ready: False
Next step: PHASE54_COMPANION_SOURCE_SEARCH_FOR_DEMOGRAPHICS_UNITS_ACQUISITION_AND_PROVENANCE


In [8]:
# CODE CELL 8/8 — Public-safe Phase53 outputs

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase53_STRUCTURAL_VALIDATION.csv',validation))
artifacts.append(save('ICHI2027_Phase53_SOURCE_TO_FHIR_PRESERVATION.csv',preservation))
artifacts.append(save('ICHI2027_Phase53_MISSINGNESS_POLICY.csv',missingness))
artifacts.append(save('ICHI2027_Phase53_FEATURE_COVERAGE.csv',coverage))

resource_counts=[
 {'resource_type':'Patient','private_resource_count':258},
 {'resource_type':'Observation','private_resource_count':4128},
 {'resource_type':'ImagingStudy','private_resource_count':0},
 {'resource_type':'Device','private_resource_count':0},
 {'resource_type':'Provenance','private_resource_count':0},
 {'resource_type':'Condition','private_resource_count':0}
]
artifacts.append(save('ICHI2027_Phase53_FHIR_RESOURCE_COUNTS.csv',resource_counts))

readme=STAGE/'ICHI2027_Phase53_README.txt'
readme.write_text(
 'Phase53 creates a private FHIR R4 Bulk-NDJSON source representation from the authorized Phase52 normalized source.\n'
 'Public artifacts contain aggregate counts and preservation/missingness evidence only; no participant IDs, private FHIR IDs, source values, original columns, private paths, or private file hashes are exported.\n'
 'FHIR Patient resources omit gender/birthDate because the source does not provide them.\n'
 'FHIR Observation resources use status=unknown, local opaque source-column coding, local anatomy/laterality coding, numeric values, and local baseline/M12 extensions.\n'
 'Units, clinical dates, ImagingStudy links, Device/model information, Provenance, and clinical MCI status are not invented.\n'
 'This is a source-side FHIR representation, not OMOP/MI-CDM execution and not a real-data fidelity result.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase52_shareable_sha256':sha256(P52),
 'source_file_token':SOURCE_TOKEN,
 'authorization_attestation_carried_forward':True,
 'private_fhir_patient_ndjson_created':True,
 'private_fhir_observation_ndjson_created':True,
 'private_fhir_manifest_created':True,
 'fhir_patient_count':258,
 'fhir_observation_count':4128,
 'fhir_imagingstudy_count':0,
 'fhir_device_count':0,
 'fhir_provenance_count':0,
 'fhir_condition_count':0,
 'all_structural_preflight_checks_passed':all(r['passed'] for r in validation),
 'source_to_fhir_person_count_preserved':True,
 'source_to_fhir_measurement_count_preserved':True,
 'source_to_fhir_numeric_values_represented':True,
 'source_to_fhir_timepoint_labels_represented':True,
 'source_to_fhir_source_column_tokens_represented':True,
 'units_invented':False,
 'clinical_dates_invented':False,
 'acquisition_ids_invented':False,
 'model_versions_invented':False,
 'provenance_invented':False,
 'clinical_mci_status_invented':False,
 'ready_for_private_fhir_to_target_preflight':True,
 'ready_for_official_armA':False,
 'next_step':next_step,
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'real_authorized_mri_processed_by_proposed_fidelity_method':False,
 'independently_blinded_external_validation':False,
 'privacy':'PUBLIC_AGGREGATE_METADATA_ONLY_PRIVATE_FHIR_IDS_VALUES_PATHS_AND_HASHES_NOT_EXPORTED',
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Phase53 proves deterministic source-to-FHIR representation with explicit missingness. It does not establish official OMOP/MI-CDM transformability or real-data fidelity performance.'
}

summary_path=STAGE/'ICHI2027_Phase53_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase53_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase53 private FHIR source representation frozen.')
print('FHIR Patients:',258)
print('FHIR Observations:',4128)
print('Official Arm A ready: False')
print('Next step:',next_step)
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase53 private FHIR source representation frozen.
FHIR Patients: 258
FHIR Observations: 4128
Official Arm A ready: False
Next step: PHASE54_COMPANION_SOURCE_SEARCH_FOR_DEMOGRAPHICS_UNITS_ACQUISITION_AND_PROVENANCE
Shareable: /content/drive/MyDrive/ICHI2027/Phase53/run_20261001T053521_159753Z/ICHI2027_Phase53_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase53/run_20261001T053521_159753Z/ICHI2027_Phase53_PUBLIC_OUTPUTS.zip


## Why the next step is a companion-source search

Phase 53 can represent the authorized measurements faithfully in FHIR, but official FHIR→OMOP/MI-CDM execution still lacks source evidence needed by the target model.

The next useful step is therefore to search the authorized local source family for **companion tables** that can legitimately supply:

- demographics required for `PERSON`;
- explicit measurement-unit evidence;
- acquisition / image / study identifiers;
- extraction software or model version;
- provenance/processing identifiers;
- clinical diagnostic status, if available and authorized.

Those fields must be joined through real source keys. They must not be manufactured from the longitudinal measurement table.
